In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Dense, BatchNormalization
from tensorflow.keras.preprocessing.image import load_img, img_to_array,ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import matplotlib.pyplot as plt
import shutil

In [8]:
train_path = 'archive/Train_data'

target_size = (150, 150, 3)

def resize_images(path, target_size):
    for subdir, dirs, files in os.walk(path):
        for file in files:
            img_path = os.path.join(subdir, file)
            with Image.open(img_path) as img:
                resized_img = img.resize(target_size[:2])  # Only use the width and height for resizing
                resized_img.save(img_path)  # Overwrite the original image

resize_images(train_path, target_size)


In [9]:
from PIL import Image

train_path = 'archive/Train_data'

target_size = (150, 150, 3)

def print_image_sizes(path):
    for subdir, dirs, files in os.walk(path):
        for file in files:
            img_path = os.path.join(subdir, file)
            with Image.open(img_path) as img:
                print(f"Image {img_path} size is {img.size}")

print_image_sizes(train_path)


Image archive/Train_data\Canine Impetigo\download (1).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download (2).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download (3).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download (4).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download (5).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download (6).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download (7).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download (8).jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\download.jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\Impetigo 1.jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\Impetigo 2.jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\Impetigo 3.jpg size is (150, 150)
Image archive/Train_data\Canine Impetigo\Impetigo 4.jpg size is (150, 150)
Image archi

In [3]:

# Define the classes
classes = ['Canine Impetigo','Canine Lupus','Feline Ringworm','Manage']

# Image dimensions
img_shape = (150, 150, 3)

# Path to your training, validation, and test datasets
train_path = 'archive/Train_data'
test_path = 'archive/Test_data'


In [5]:
def load_and_preprocess_image(img_path):
    try:
        img = load_img(img_path, target_size=(img_shape[0], img_shape[1]))
        img_array = img_to_array(img)

        img_array /= 255.0  # Normalize to [0,1]
        return img_array
    except Exception as e:
        print(f"Error loading image {img_path}: {str(e)}")
        return None

def generator_fn(image_paths, labels, batch_size=32):
    num_samples = len(image_paths)
    while True:
        indices = np.random.choice(num_samples, batch_size, replace=False)
        batch_image_paths = [image_paths[i] for i in indices]
        batch_labels = [labels[i] for i in indices]

        # Load and preprocess the batch of images
        batch_images = [load_and_preprocess_image(img_path) for img_path in batch_image_paths]

        # Skip None values (failed to load)
        batch_images = [img for img in batch_images if img is not None]

        # Skip batch if all images failed to load
        if not batch_images:
            continue

        # One-hot encode the labels
        batch_labels_one_hot = tf.keras.utils.to_categorical(batch_labels, len(classes))

        # Check if the batch size matches the specified batch_size
        if len(batch_images) < batch_size:
            # If the last batch is smaller, pad it with additional samples
            extra_samples = batch_size - len(batch_images)
            extra_indices = np.random.choice(num_samples, extra_samples, replace=False)
            extra_batch_image_paths = [image_paths[i] for i in extra_indices]
            extra_batch_labels = [labels[i] for i in extra_indices]
            extra_batch_images = [load_and_preprocess_image(img_path) for img_path in extra_batch_image_paths]
            extra_batch_images = [img for img in extra_batch_images if img is not None]
            batch_images.extend(extra_batch_images)
            batch_labels.extend(extra_batch_labels)

        yield np.array(batch_images), np.array(batch_labels_one_hot)



In [6]:

# Initialize the model
model = models.Sequential()


In [7]:
# Convolutional layers with dilation
model.add(layers.Conv2D(32, (3, 3), activation='relu', dilation_rate=(2, 2), input_shape=img_shape))
model.add(layers.Conv2D(64, (3, 3), dilation_rate=(2, 2)))

# MaxPooling with stride
model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

# Dropout
model.add(layers.Dropout(0.25))

# More Convolutional layers with dilation
model.add(layers.Conv2D(128, (3, 3), activation='relu', dilation_rate=(2, 2)))
model.add(layers.Conv2D(256, (3, 3), activation='relu', dilation_rate=(2, 2)))

# MaxPooling with stride
model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

# Dropout
model.add(layers.Dropout(0.25))

# Another Convolutional layer
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# MaxPooling with stride
model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

# Flatten layer to feed into densely connected layers
model.add(layers.Flatten())

# Dense layers
model.add(layers.Dense(1024, activation='relu'))
model.add(layers.Dropout(0.3))  # Adding dropout for regularization
model.add(layers.Dense(len(classes), activation='softmax'))




In [9]:
# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [10]:
# Display the model summary
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 146, 146, 32)      896       
                                                                 
 conv2d_1 (Conv2D)           (None, 142, 142, 64)      18496     
                                                                 
 max_pooling2d (MaxPooling2  (None, 71, 71, 64)        0         
 D)                                                              
                                                                 
 dropout (Dropout)           (None, 71, 71, 64)        0         
                                                                 
 conv2d_2 (Conv2D)           (None, 67, 67, 128)       73856     
                                                                 
 conv2d_3 (Conv2D)           (None, 63, 63, 256)       295168    
                                                        

In [11]:
# Using ImageDataGenerator for data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)


train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=(img_shape[0], img_shape[1]),
    batch_size=32,
    class_mode='categorical',
    classes=classes
)



print("Number of classes:", len(train_generator.class_indices))
print("Number of training samples:", train_generator.samples)



Found 135 images belonging to 4 classes.


Number of classes: 4
Number of training samples: 135


In [12]:
# Training the model
checkpoint = ModelCheckpoint('archive/best_model.h5', monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
callbacks_list = [checkpoint]


In [13]:
history = model.fit(
    generator_fn(train_generator.filepaths, train_generator.labels),
    steps_per_epoch=len(train_generator),
    epochs=20,
    callbacks=callbacks_list
)

Epoch 1/20
5/5 [==============================] - 45s 9s/step - loss: 3.5220 - accuracy: 0.4125
Epoch 2/20
5/5 [==============================] - 39s 7s/step - loss: 1.1144 - accuracy: 0.4000
Epoch 3/20
5/5 [==============================] - 30s 6s/step - loss: 1.0956 - accuracy: 0.4313
Epoch 4/20
5/5 [==============================] - 27s 5s/step - loss: 1.0818 - accuracy: 0.4187
Epoch 5/20
5/5 [==============================] - 30s 6s/step - loss: 1.0468 - accuracy: 0.4500
Epoch 6/20
5/5 [==============================] - 26s 5s/step - loss: 1.0955 - accuracy: 0.3812
Epoch 7/20
5/5 [==============================] - 27s 5s/step - loss: 1.0745 - accuracy: 0.4750
Epoch 8/20
5/5 [==============================] - 26s 5s/step - loss: 1.0782 - accuracy: 0.4563
Epoch 9/20
5/5 [==============================] - 26s 5s/step - loss: 1.0403 - accuracy: 0.4563
Epoch 10/20
5/5 [==============================] - 32s 7s/step - loss: 1.0175 - accuracy: 0.4812
Epoch 11/20
5/5 [======================

In [15]:

save_path = 'archive/best_model.h5'



# Save the trained model
model.save(save_path)
